# Lab 26-03 · Cost Stacks, Break-Even and the Price of a Slip

Companion to [26-03](../../chapters/26-03-engineering-economics-1.md). Write each prediction down before running its cell.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import econ as e

## Part 1 · Cost waterfall

*Predict:* how much does COGS rise if final-test yield falls from 98% to 90%?

In [2]:
dpw, y = e.dies_per_wafer(300, 100), math.exp(-0.1)
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, yt in enumerate((0.98, 0.90)):
    c = e.cost_stack(17000, dpw, y, 4.0, 0.99, 1.20, yt, 1.20)
    steps = [c['die'], c['die'] + 4.0, c['after_asm'], c['after_asm'] + 1.2, c['after_test'], c['cogs']]
    print(f"final-test yield {yt:.0%}: COGS ${c['cogs']:.2f}, GM {100 * e.gross_margin(60, c['cogs']):.1f} %")
    ax.plot(range(len(steps)), steps, marker='o', label=f'final-test yield {yt:.0%}')
ax.set_xticks(range(6), ['die', '+pkg', '/assembly yield', '+test', '/test yield', '+royalty'], rotation=20)
ax.set_ylabel('cumulative cost per good unit ($)'); ax.legend()
plt.show()

final-test yield 98%: COGS $36.81, GM 38.7 %
final-test yield 90%: COGS $39.97, GM 33.4 %


## Part 2 · Volume

*Predict:* the lifetime volume at which $60 M of NRE falls below $2 a unit.

In [3]:
vol = np.linspace(1e6, 50e6, 200)
cogs = e.cost_stack(17000, dpw, y, 4.0, 0.99, 1.20, 0.98, 1.20)['cogs']
loaded = cogs + 60e6 / vol
print(f"NRE below $2 per unit above {60e6 / 2 / 1e6:.0f} million units")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(vol / 1e6, 100 * (60 - loaded) / 60)
ax.axhline(100 * (60 - cogs) / 60, color='k', ls=':', lw=1)
ax.set_xlabel('lifetime volume (million units)'); ax.set_ylabel('fully loaded margin (%)')
plt.show()

NRE below $2 per unit above 30 million units


## Part 3 · Slip

*Predict:* the NPV after a six-month slip, with the final year's sales prorated at the window.

In [4]:
contrib = 60 - cogs
units = [1e6, 3e6, 4e6, 2e6]
slips = np.linspace(0, 1, 13)
vals = []
for s in slips:
    f, t = e.program_flows([30e6, 30e6], units, contrib, slip=s, window_end=5)
    vals.append(e.npv(0.12, f, t) / 1e6)
print(f"NPV: no slip {vals[0]:.1f}, 6 months {vals[6]:.1f}, 12 months {vals[-1]:.1f} ($M)")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(12 * slips, vals, marker='o')
ax.set_xlabel('slip (months)'); ax.set_ylabel('NPV at 12% ($M)')
plt.show()

NPV: no slip 96.5, 6 months 75.6, 12 months 56.6 ($M)


## Part 4 · Tornado

*Predict:* the order of the bars for ±10% in ASP, volume, COGS and NRE.

In [5]:
def model(p):
    f, t = e.program_flows([p['nre'] / 2] * 2, [p['vol'] * u for u in (1, 3, 4, 2)], p['asp'] - p['cogs'])
    return e.npv(0.12, f, t) / 1e6

base = {'asp': 60.0, 'cogs': cogs, 'vol': 1e6, 'nre': 60e6}
rows = e.tornado(base, model)
b0 = model(base)
fig, ax = plt.subplots(figsize=(6, 3))
for k, (name, lo, hi) in enumerate(rows):
    ax.barh(k, hi - b0, left=b0, color='tab:blue'); ax.barh(k, lo - b0, left=b0, color='tab:orange')
    print(f"{name:5s}: {lo:6.1f} to {hi:6.1f} $M")
ax.set_yticks(range(len(rows)), [r[0] for r in rows]); ax.invert_yaxis(); ax.set_xlabel('NPV ($M)')
plt.show()

asp  :   56.9 to  136.2 $M
cogs :  120.8 to   72.2 $M
vol  :   81.2 to  111.9 $M
nre  :  102.2 to   90.8 $M
